# Data Wrangling - Groupby, Pivot and Join

In [1]:
import numpy as np
import pandas as pd

In [2]:
modelDF = pd.DataFrame({"model": ["sentra", "yaris", "armada",
                              "corolla", "veloster", "highlander",
                              "edge", "cherokee", "lancer"],
                     "cant": [4, 3, 12, 6, 7, 8, 3, 5, 6]})
modelDF

,model,cant
0,sentra,4
1,yaris,3
2,armada,12
3,corolla,6
4,veloster,7
5,highlander,8
6,edge,3
7,cherokee,5
8,lancer,6


In [3]:
model_to_brand = {
  "sentra": "nissan",
  "yaris": "toyota",
  "armada": "nissan",
  "corolla": "toyota",
  "veloster": "hyundai",
  "cherokee": "jeep",
    "lancer": "mitsubishi",
    "edge": "ford",
    "highlander": "toyota"
}

In [4]:
modelDF["brand"] = modelDF["model"].map(model_to_brand)
modelDF

# Recuerde: un "mapping" es una función. Si 'model' no tiene un 'brand' al que asignarse, usará NaN. 
# Si por el contrario, hay 'brand'sin 'model,  no lo incluirá en el df.

,model,cant,brand
0,sentra,4,nissan
1,yaris,3,toyota
2,armada,12,nissan
3,corolla,6,toyota
4,veloster,7,hyundai
5,highlander,8,toyota
6,edge,3,ford
7,cherokee,5,jeep
8,lancer,6,mitsubishi


In [5]:
modelDF.groupby('brand').sum()

,cant
brand,
ford,3
hyundai,7
jeep,5
mitsubishi,6
nissan,16
toyota,17


In [6]:
# Pivot permite elegir una de las columnas para ser usadas como índice y cambiar las columnas.
# Con ello, también se agrupa.
modelDF.pivot(index='brand',columns='model')

cant                                                          \
model      armada cherokee corolla edge highlander lancer sentra veloster   
brand                                                                       
ford          NaN      NaN     NaN  3.0        NaN    NaN    NaN      NaN   
hyundai       NaN      NaN     NaN  NaN        NaN    NaN    NaN      7.0   
jeep          NaN      5.0     NaN  NaN        NaN    NaN    NaN      NaN   
mitsubishi    NaN      NaN     NaN  NaN        NaN    6.0    NaN      NaN   
nissan       12.0      NaN     NaN  NaN        NaN    NaN    4.0      NaN   
toyota        NaN      NaN     6.0  NaN        8.0    NaN    NaN      NaN   

                  
model      yaris  
brand             
ford         NaN  
hyundai      NaN  
jeep         NaN  
mitsubishi   NaN  
nissan       NaN  
toyota       3.0

In [7]:
data = pd.DataFrame({"col1": ["one", "two"] * 3 + ["two"]*2,
                     "col2": [1, 1, 2, 3, 3, 4, 4,4]})
data

,col1,col2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4
7,two,4


In [8]:
data["col3"] = range(8)

In [9]:
data = data.drop_duplicates(subset=["col1", "col2"], keep="last")
data

,col1,col2,col3
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
7,two,4,7


In [10]:
data.pivot(index = 'col1', columns= 'col2', values = 'col3')

col2,1,2,3,4
col1,,,,
one,0.0,2.0,4.0,NaN
two,1.0,NaN,3.0,7.0


## Join

El método join añade las columnas de diferentes data frames unas al lado de las otras. Sin embargo, si las columnas no son iguales en longitud las llenará con NaN. A diferencia del merge que usa nombres x y y por defecto en las columnas, en el join debo especificar los nombres que quiero para las nuevas columnas, en caso de que tengan los mismos nombres en ambos dataframes.

Puedo unir usando como criterio de unión:
- los índices
- el solape de las columnas

In [11]:
brandDF = pd.DataFrame(["nissan","toyota","hyundai","jeep","mitsubishi","ford"], columns = ['brand2'])
brandDF

,brand2
0,nissan
1,toyota
2,hyundai
3,jeep
4,mitsubishi
5,ford


In [12]:
# Ejemplo 1: Union de dos datas frames usando los índices para unir. Si columnas de 'brand' tuvieran el mismo nombre, 
# debo especificar los sufijos
modelDF.join(brandDF)

# Note que esta 'union' no asigna(no hace mapping). Simplemente pone una columna al lado de la otra en orden. 

,model,cant,brand,brand2
0,sentra,4,nissan,nissan
1,yaris,3,toyota,toyota
2,armada,12,nissan,hyundai
3,corolla,6,toyota,jeep
4,veloster,7,hyundai,mitsubishi
5,highlander,8,toyota,ford
6,edge,3,ford,NaN
7,cherokee,5,jeep,NaN
8,lancer,6,mitsubishi,NaN


In [13]:
brandDF.columns = ['brand']
brandDF

,brand
0,nissan
1,toyota
2,hyundai
3,jeep
4,mitsubishi
5,ford


In [14]:
modelDF.join(brandDF,lsuffix='_leftDF', rsuffix='_rightDF')

,model,cant,brand_leftDF,brand_rightDF
0,sentra,4,nissan,nissan
1,yaris,3,toyota,toyota
2,armada,12,nissan,hyundai
3,corolla,6,toyota,jeep
4,veloster,7,hyundai,mitsubishi
5,highlander,8,toyota,ford
6,edge,3,ford,NaN
7,cherokee,5,jeep,NaN
8,lancer,6,mitsubishi,NaN


In [15]:
# Si ambos dataframes tienen una columna compartida (mismo nombre y data relacionada), puedo usar esa columna 
# para unir. En ese caso, esa columna se debe convertir en los índices. 
modelCOPY = modelDF.set_index('brand').join(brandDF.set_index('brand'))

In [16]:
# Groupby igualmente los agrupa pero la suma esta vez en en el axis=0 (sumar fila mas fila). Así que tenemos el mismo resultado.
modelCOPY.groupby('brand').sum()

,cant
brand,
ford,3
hyundai,7
jeep,5
mitsubishi,6
nissan,16
toyota,17


In [19]:
modelCOPY

,model,cant
brand,,
ford,edge,3
hyundai,veloster,7
jeep,cherokee,5
mitsubishi,lancer,6
nissan,sentra,4
nissan,armada,12
toyota,yaris,3
toyota,corolla,6
toyota,highlander,8


In [20]:
# Vea lo que pasa cuando hago "slice" a los nuevos índices (usando loc y no iloc porque son labels y no enteros)
modelCOPY.loc['ford']
modelCOPY.loc['nissan']
modelCOPY.loc['toyota']

,model,cant
brand,,
toyota,yaris,3
toyota,corolla,6
toyota,highlander,8


In [35]:
brandDF.loc[len(brandDF.index)] = ['Mercedes']
brandDF

,brand
0,nissan
1,toyota
2,hyundai
3,jeep
4,mitsubishi
5,ford
6,Mercedes


In [42]:
# Note que si en el dataframe que se está anejando con 'join' tiene algún elemento en la columna que se usa para
# unir que no está en el dataframe que invoca, este elemento no se incluye.
modelDF.set_index('brand').join(brandDF.set_index('brand'))

,model,cant
brand,,
ford,edge,3
hyundai,veloster,7
jeep,cherokee,5
mitsubishi,lancer,6
nissan,sentra,4
nissan,armada,12
toyota,yaris,3
toyota,corolla,6
toyota,highlander,8


In [43]:
modelDF.join(brandDF.set_index('brand'), on = 'brand')

,model,cant,brand
0,sentra,4,nissan
1,yaris,3,toyota
2,armada,12,nissan
3,corolla,6,toyota
4,veloster,7,hyundai
5,highlander,8,toyota
6,edge,3,ford
7,cherokee,5,jeep
8,lancer,6,mitsubishi
